# Day 62: Neural Network Fundamentals
## Layers, Activations & Building Deeper Networks

---

# PART 1: THEORY

## 1. Anatomy of a Neural Network

```
[Input Layer] -> [Hidden Layer 1] -> [Hidden Layer 2] -> [Output Layer]
  (features)       (learns patterns)   (learns abstractions)   (predictions)
```

Each layer is a group of neurons. Information flows FORWARD (input -> hidden -> output). This is called a **feedforward neural network**.

**Key components:**
- **Input Layer:** Size = number of features in your data. Does NOT count as a "layer" (no computation)
- **Hidden Layers:** Where learning happens. More layers = "deeper" network
- **Output Layer:** Produces final predictions
- **Weights:** Numbers the network adjusts during training
- **Bias:** Offset added to each neuron's weighted sum

## 2. Activation Functions — The Decision Makers

Without activation functions, a neural network is just a linear model — no matter how deep!

| Function | Equation | Range | Best Used In |
|----------|----------|-------|-------------|
| **Sigmoid** | 1/(1+e^{-x}) | (0, 1) | Output for binary classification |
| **Tanh** | (e^x-e^{-x})/(e^x+e^{-x}) | (-1, 1) | Rarely used now |
| **ReLU** | max(0, x) | [0, inf) | **Hidden layers (DEFAULT)** |
| **Leaky ReLU** | max(0.01x, x) | (-inf, inf) | When ReLU "dies" |
| **Softmax** | e^{x_i}/sum(e^{x}) | (0, 1) | Output for multi-class |

**Why ReLU is the default:**
- Fast to compute (just max)
- Doesn't saturate like sigmoid (gradient stays strong)
- Works well empirically

**The "Dying ReLU" problem:** If a neuron always outputs 0, its gradient is 0 — it stops learning forever. Leaky ReLU fixes this.

## 3. How Many Layers and Neurons?

There's no exact formula — it's part science, part art.

**Guidelines:**
- Start with 1-2 hidden layers
- Number of neurons: between input size and output size
- Powers of 2 are common: 32, 64, 128, 256, 512
- If underfitting: add more neurons/layers
- If overfitting: reduce neurons/layers or add dropout

**The Universal Approximation Theorem:** A network with ONE hidden layer can approximate ANY continuous function — given enough neurons. But "enough" might be millions!

**In practice:** Deep (many layers) > Wide (many neurons per layer) for the same parameter count.

## 4. The Math of One Neuron

```
z = w1*x1 + w2*x2 + ... + wn*xn + b   # Weighted sum
a = activation(z)                        # Apply activation
```

**Example:** Predict if a student passes based on study hours (x1) and sleep hours (x2):
```
z = 0.7 * study_hours + 0.3 * sleep_hours - 2.0
If z > 0 -> pass, else -> fail
```

---

# PART 2: PRACTICAL

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import make_classification, make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


## 5. Visualizing Activation Functions

In [ ]:
x = np.linspace(-6, 6, 300)

def sigmoid(x): return 1/(1+np.exp(-x))
def relu(x): return np.maximum(0, x)
def leaky_relu(x, alpha=0.1): return np.where(x > 0, x, alpha * x)
def tanh(x): return np.tanh(x)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
funcs = [('Sigmoid', sigmoid, 'b'), ('Tanh', tanh, 'r'),
         ('ReLU', relu, 'g'), ('Leaky ReLU (a=0.1)', leaky_relu, 'purple')]

for (name, fn, color), ax in zip(funcs, axes.flatten()):
    ax.plot(x, fn(x), color=color, linewidth=3)
    ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
    ax.axvline(x=0, color='black', linewidth=0.5, linestyle='--')
    ax.set_title(name, fontsize=14)
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.grid(True, alpha=0.3)

plt.suptitle('Activation Functions Compared', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Building Deeper Networks — Compare Architectures

In [ ]:
# Generate a non-linear classification problem
X, y = make_moons(n_samples=1000, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlGn', alpha=0.6, edgecolors='k', linewidth=0.3)
plt.title('Moons Dataset — NOT linearly separable!')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()


In [ ]:
# Compare architectures on the same problem
architectures = {
    'Shallow [1 layer, 4 neurons]': [layers.Dense(4, activation='relu')],
    'Medium [2 layers, 16->8]': [layers.Dense(16, activation='relu'), layers.Dense(8, activation='relu')],
    'Deep [3 layers, 64->32->16]': [layers.Dense(64, activation='relu'), layers.Dense(32, activation='relu'), layers.Dense(16, activation='relu')],
    'Deep + Dropout': [layers.Dense(64, activation='relu'), layers.Dropout(0.3), layers.Dense(32, activation='relu'), layers.Dropout(0.3), layers.Dense(16, activation='relu')],
}

results = []
for name, hidden_layers in architectures.items():
    model = keras.Sequential(hidden_layers + [layers.Dense(1, activation='sigmoid')])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    history = model.fit(X_train_s, y_train, epochs=100, validation_split=0.2, verbose=0)
    _, train_acc = model.evaluate(X_train_s, y_train, verbose=0)
    _, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
    results.append({'Architecture': name, 'Train Acc': train_acc, 'Test Acc': test_acc, 'Params': model.count_params()})

for r in results:
    print(f"{r['Architecture']:35s} | Train: {r['Train Acc']:.3f} | Test: {r['Test Acc']:.3f} | Params: {r['Params']:,}")


## 7. Effect of Layer Width vs Depth

In [ ]:
# Same total parameters, different arrangement
# Wide (1 layer, 100 neurons) vs Deep (3 layers, small)
X, y = make_classification(n_samples=2000, n_features=20, n_informative=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)

configs = {
    'Wide: 1 layer x 128': [layers.Dense(128, activation='relu')],
    'Deep: 3 layers x 64->32->16': [layers.Dense(64, activation='relu'), layers.Dense(32, activation='relu'), layers.Dense(16, activation='relu')],
    'Very Deep: 5 layers x 32->...': [layers.Dense(32, activation='relu') for _ in range(5)],
}

for name, h in configs.items():
    m = keras.Sequential(h + [layers.Dense(1, activation='sigmoid')])
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    m.fit(X_tr_s, y_tr, epochs=50, validation_split=0.2, verbose=0)
    _, acc = m.evaluate(X_te_s, y_te, verbose=0)
    print(f"{name:40s} | Test Acc: {acc:.3f} | Params: {m.count_params():,}")


---

# PART 3: EXERCISES

In [ ]:
# Exercise 1: Find the best architecture for the circles dataset
X_c, y_c = make_circles(n_samples=1000, noise=0.1, factor=0.5, random_state=42)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)
scaler_c = StandardScaler()
Xc_tr_s = scaler_c.fit_transform(Xc_tr)
Xc_te_s = scaler_c.transform(Xc_te)

# Try at least 3 different architectures
# The circles dataset is harder than moons — you'll need more layers!
print("Try different architectures on the circles dataset!")
print("Hint: Start with [64, 32, 16] and adjust")


In [ ]:
# Exercise 2: Compare activation functions on the same architecture
X_m, y_m = make_moons(n_samples=1000, noise=0.2, random_state=42)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(X_m, y_m, test_size=0.2, random_state=42)
scaler_m = StandardScaler()
Xm_tr_s = scaler_m.fit_transform(Xm_tr)
Xm_te_s = scaler_m.transform(Xm_te)

activations = ['relu', 'tanh', 'sigmoid', 'elu']
for act in activations:
    m = keras.Sequential([
        layers.Dense(16, activation=act, input_shape=(2,)),
        layers.Dense(8, activation=act),
        layers.Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    m.fit(Xm_tr_s, ym_tr, epochs=50, validation_split=0.2, verbose=0)
    _, acc = m.evaluate(Xm_te_s, ym_te, verbose=0)
    print(f"Activation {act:10s} -> Test Acc: {acc:.3f}")


## Key Takeaways

- **ReLU** is the default activation for hidden layers
- **Sigmoid** for binary output, **Softmax** for multi-class
- **Deeper** networks learn more abstract features
- **Dropout** prevents overfitting by randomly disabling neurons
- More layers > more neurons per layer (for same param count)
- Always **scale** your data before feeding to a neural network
- The architecture (layers + neurons) is a hyperparameter you tune

**Tomorrow:** Training — loss functions, backpropagation, optimizers, and how to avoid overfitting!